<a href="https://colab.research.google.com/github/Mennaali30/recipe-agent/blob/main/notebooks/run_in_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1) Clone and install (Python 3.13; if Colab asks to restart the session, restart and continue)
!git clone https://github.com/Mennaali30/recipe-agent.git
%cd recipe-agent
%pip install -q -r requirements.txt

In [ ]:
!find / -name recipes.parquet -path "*processed*" 2>/dev/null

In [ ]:
# 2) Build the data and the index (once per session; takes a few minutes on CPU)
import sys, kagglehub
sys.path.insert(0, "/content/recipe-agent/src")
from recipe_agent.ingest import run
from recipe_agent.retriever import build_index

path = kagglehub.dataset_download("irkaal/foodcom-recipes-and-reviews")
run(raw_dir=path)
print("indexed:", build_index())

In [ ]:
# 3) Your own Gemini key goes in Colab Secrets as GOOGLE_API_KEY (never paste it here)
import os
from google.colab import userdata

try:
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    print("No GOOGLE_API_KEY secret: only the saved example answers will work.")
os.environ["LLM_PROVIDER"] = "gemini"
os.environ["GEMINI_MODEL"] = "gemini-3.6-flash"            # check the current model names in Google AI Studio
os.environ["GEMINI_FALLBACK_MODELS"] = "gemini-3.8-flash"

from recipe_agent import graph
print(graph.answer("عندي بطاطس وبيض وعندي حساسية من اللبن"))   # saved example: no model call

In [ ]:
# 4) Chat UI (the public link has no login and uses your quota: do not share it)
sys.path.insert(0, "/content/recipe-agent/app")
import gradio_app
gradio_app.demo.launch(share=True)

In [ ]:
!rm -rf /tmp/check_repo
!git clone -q https://github.com/Mennaali30/recipe-agent.git /tmp/check_repo
import json
nb = json.load(open("/tmp/check_repo/notebooks/run_in_colab.ipynb", encoding="utf-8"))
cells = nb["cells"]
print("cells:", len(cells), "| with outputs:", sum(1 for c in cells if c.get("outputs")))
print("key-like text:", "AIza" in json.dumps(nb))
for c in cells:
    print("---", "".join(c["source"])[:60].replace("\n", " "))

In [ ]:
out = graph.run_agent("الجو عامل إيه النهارده؟", user_id="t")
print(out["request"].model_dump())
print(out["answer"])

In [ ]:
import os, importlib
import pandas as pd
from recipe_agent import evaluation as ev
importlib.reload(ev)

COLS = ["text", "pred_ingredients", "pred_exclude", "pred_minutes", "pred_language", "all_ok", "error"]

os.environ["GEMINI_MODEL"] = "gemini-3.6-flash"
os.environ["GEMINI_FALLBACK_MODELS"] = "gemini-3.8-flash"
df1 = ev.eval_intent(cases=ev.INTENT_CASES[:9], tries=2, base_wait=30)

os.environ["GEMINI_MODEL"] = "gemini-3.8-flash"
os.environ["GEMINI_FALLBACK_MODELS"] = "gemini-3.6-flash"
df2 = ev.eval_intent(cases=ev.INTENT_CASES[9:], tries=2, base_wait=30)

df_int = pd.concat([df1, df2], ignore_index=True)
print(df_int[[c for c in COLS if c in df_int.columns]].to_string())
print(ev.summarize_intent(df_int))

In [ ]:
import importlib
from recipe_agent import evaluation as ev, tools
importlib.reload(tools)
importlib.reload(ev)

df_ret = ev.eval_retrieval()
print(ev.summarize_retrieval(df_ret))
df_all, leaks = ev.eval_allergy_audit()
print(df_all[["allergen", "audit_flagged", "filter_flagged", "leaked"]].to_string())